# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [1]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


BRANCH = "ts"  # 본인이 사용할 GitHub 브랜치명


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url

    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, clone_url, str(repo_dir)],
            check=True,
        )
        subprocess.run(
            ["git", "remote", "set-url", "origin", repo_url],
            cwd=repo_dir,
            check=True,
        )
    else:
        print(f"이미 clone된 저장소를 업데이트합니다: {repo_dir}")
        subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)

        local_branches = subprocess.run(
            ["git", "branch", "--list", BRANCH],
            cwd=repo_dir,
            text=True,
            capture_output=True,
            check=True,
        ).stdout.strip()

        if local_branches:
            subprocess.run(["git", "switch", BRANCH], cwd=repo_dir, check=True)
        else:
            subprocess.run(
                ["git", "switch", "-c", BRANCH, "--track", f"origin/{BRANCH}"],
                cwd=repo_dir,
                check=True,
            )

        subprocess.run(
            ["git", "pull", "--ff-only", "origin", BRANCH],
            cwd=repo_dir,
            check=True,
        )

    os.chdir(repo_dir)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        cwd=repo_dir,
        check=True,
    )

else:
    repo_dir = Path(".").resolve()


src_path = str(repo_dir / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Repo: {repo_dir}")
print(f"Branch: {BRANCH}")

subprocess.run(["git", "branch", "--show-current"], cwd=repo_dir, check=True)
subprocess.run(["git", "log", "-1", "--oneline", "--decorate"], cwd=repo_dir, check=True)

GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): github.com/liamyoum/W14-gpt-lab.git
GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ··········
Repo: /content/W14-gpt-lab
Branch: ts


CompletedProcess(args=['git', 'log', '-1', '--oneline', '--decorate'], returncode=0)

In [2]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [3]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

다운로드 중: https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
저장됨: /content/W14-gpt-lab/data/ratings_train.txt
다운로드 중: https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt
저장됨: /content/W14-gpt-lab/data/ratings_test.txt
사전 학습 train 텍스트: /content/W14-gpt-lab/data/nsmc_lm_train.txt (1,379,486자)
사전 학습 val 텍스트: /content/W14-gpt-lab/data/nsmc_lm_val.txt (120,560자)
감성 분류 train: /content/W14-gpt-lab/data/nsmc_sentiment_train.jsonl (137,996개)
감성 분류 val: /content/W14-gpt-lab/data/nsmc_sentiment_val.jsonl (11,999개)
감성 분류 test: /content/W14-gpt-lab/data/nsmc_sentiment_test.jsonl (49,997개)
LM train exists: True /content/W14-gpt-lab/data/nsmc_lm_train.txt
LM val exists: True /content/W14-gpt-lab/data/nsmc_lm_val.txt


In [4]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

train chars: 1379486
val chars: 120560
개재미없다. 감독의 연출력의 한계
이제서야 보게된 대 명작 연출미가 정말 훌륭하다!!!!!!!!
소주미라클을 만들어라
귀여운 캐릭터들도 많이 나와서 보러 가야 겠어요..
블랙 코미디가 싫어요.
평점깎고싶다10글자
TV시리즈가 너무재밌어서 영화는 기대안하고 봤는데 역시....최고네요
개인적 공감이 글쎄?
시작은 니시지마 때문에 봤는데 나름 괜찮은 영화 봤다고


## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [5]:
run_pytest("tests/test_bpe.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_bpe.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W14-gpt-lab
plugins: typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
collecting ... collected 6 items

tests/test_bpe.py::TestSpecialTokens::test_special_ids_fixed PASSED      [ 16%]
tests/test_bpe.py::TestBPETokenizer::test_init_special_tokens PASSED     [ 33%]
tests/test_bpe.py::TestBPETokenizer::test_save_load_restores_vocab PASSED [ 50%]
tests/test_bpe.py::TestBPETokenizer::test_encode_decode_restores_original_text PASSED [ 66%]
tests/test_bpe.py::TestBPETokenizer::test_get_special_ids PASSED         [ 83%]
tests/test_bpe.py::TestBPETrain::test_train_increases_vocab PASSED       [100%]

============================== 6 passed in 0.03s ===============================


선택한 테스트를 통과했습니다.


0

In [ ]:
# BPE 구현 후 작은 말뭉치로 인코딩/디코딩 복원을 확인합니다.
# try:
#     from bpe import BPETokenizer

#     tokenizer = BPETokenizer(vocab_size=1000)
#     tokenizer.train(corpus[:100000])
#     sample = "이 영화는 정말 좋았다! English 123"
#     ids = tokenizer.encode(sample, add_bos_eos=True)
#     print(ids[:20])
#     print(tokenizer.decode(ids))
# except NotImplementedError as e:
#     print("BPE TODO 미구현:", e)

KeyboardInterrupt: 

### Google Drive에 어휘사전 json이 없다면 새로 생성, 있다면 load 하는 Code Block

In [15]:
from pathlib import Path
import time
import sys
import platform
import importlib
import shutil

from google.colab import drive

drive.mount("/content/drive")

# 이미 import된 예전 bpe.py가 있으면 제거
sys.modules.pop("bpe", None)
importlib.invalidate_caches()

from bpe import BPETokenizer


CORPUS_SIZE = 1_500_000
VOCAB_SIZE = 3000 # 어휘 사전 크기 변경하고 싶다면 여기 변경

repo_vocab_path = Path("data/nsmc_bpe_vocab_3000.json") # 여기도 숫자 변경
drive_vocab_dir = Path("/content/drive/MyDrive/W14-gpt-lab/data")
drive_vocab_dir.mkdir(parents=True, exist_ok=True)

drive_vocab_path = drive_vocab_dir / "nsmc_bpe_vocab_3000.json" # 어휘사전 크기에 따라 load를 다르게 하고 싶다면 여기 변경

text = corpus[:CORPUS_SIZE]
tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE)

if drive_vocab_path.exists():
    start = time.perf_counter()
    tokenizer.load(drive_vocab_path)
    elapsed = time.perf_counter() - start
    mode = "loaded_from_drive"
else:
    start = time.perf_counter()
    tokenizer.train(text)
    elapsed = time.perf_counter() - start

    tokenizer.save(drive_vocab_path)
    tokenizer.save(repo_vocab_path)
    mode = "trained_and_saved"

# repo data에도 복사해두기
if drive_vocab_path.exists():
    repo_vocab_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_vocab_path, repo_vocab_path)

sample = "이 영화는 정말 좋았다! English 123"
ids = tokenizer.encode(sample, add_bos_eos=True)
decoded = tokenizer.decode(ids)

env = "Colab"
try:
    import torch
    device = "GPU" if torch.cuda.is_available() else "CPU"
except ImportError:
    device = "CPU"

print("mode:", mode)
print("environment:", env, device)
print("python:", platform.python_version())
print("corpus_size chars:", len(text))
print("corpus_size bytes:", len(text.encode("utf-8")))
print("vocab_size:", VOCAB_SIZE)
print("actual_vocab_size:", len(tokenizer.id_to_token))
print("training_or_load_time_sec:", round(elapsed, 2))
print("training_or_load_time_min:", round(elapsed / 60, 2))
print("drive_vocab_path:", drive_vocab_path)
print("repo_vocab_path:", repo_vocab_path)
print("decode_ok:", decoded == sample)
print("sample_ids:", ids[:20])
print("decoded:", decoded)

Mounted at /content/drive
mode: loaded_from_drive
environment: Colab GPU
python: 3.12.13
corpus_size chars: 1379486
corpus_size bytes: 3335336
vocab_size: 3000
actual_vocab_size: 3000
training_or_load_time_sec: 0.3
training_or_load_time_min: 0.01
drive_vocab_path: /content/drive/MyDrive/W14-gpt-lab/data/nsmc_bpe_vocab_3000.json
repo_vocab_path: data/nsmc_bpe_vocab_3000.json
decode_ok: True
sample_ids: [2, 267, 1003, 532, 860, 1996, 36, 73, 114, 107, 112, 109, 119, 108, 505, 54, 55, 3]
decoded: 이 영화는 정말 좋았다! English 123


## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [7]:
run_pytest("tests/test_dataset.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_dataset.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W14-gpt-lab
plugins: typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
collecting ... collected 4 items

tests/test_dataset.py::TestGPTDataset::test_dataset_length PASSED        [ 25%]
tests/test_dataset.py::TestGPTDataset::test_dataset_getitem_shape PASSED [ 50%]
tests/test_dataset.py::TestCreateDataloader::test_dataloader_batch_shape PASSED [ 75%]
tests/test_dataset.py::TestInputEmbedding::test_input_embedding_shape PASSED [100%]

============================== 4 passed in 4.73s ===============================


선택한 테스트를 통과했습니다.


0

In [ ]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    small_tokenizer = BPETokenizer(vocab_size=300)
    small_tokenizer.train(corpus[:5000])
    token_ids = small_tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=300, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)

torch.Size([2, 32]) torch.Size([2, 32]) torch.Size([2, 32, 32])


## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [9]:
run_pytest("tests/test_attention.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_attention.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W14-gpt-lab
plugins: typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
collecting ... collected 2 items

tests/test_attention.py::TestMultiHeadAttention::test_mha_output_shape PASSED [ 50%]
tests/test_attention.py::TestMultiHeadAttention::test_mha_causal_mask_future_zero PASSED [100%]

============================== 2 passed in 1.67s ===============================


선택한 테스트를 통과했습니다.


0

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [10]:
run_pytest("tests/test_model.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_model.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W14-gpt-lab
plugins: typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
collecting ... collected 7 items

tests/test_model.py::TestLayerNorm::test_layernorm_shape PASSED          [ 14%]
tests/test_model.py::TestGELU::test_gelu_shape PASSED                    [ 28%]
tests/test_model.py::TestFeedForward::test_feedforward_shape PASSED      [ 42%]
tests/test_model.py::TestTransformerBlock::test_transformer_block_shape PASSED [ 57%]
tests/test_model.py::TestGPTModel::test_gpt_forward_shape PASSED         [ 71%]
tests/test_model.py::TestGPTModel::test_gpt_forward_with_targets_returns_loss PASSED [ 85%]
tests/test_model.py::TestGenerateTextSimple::test_generate_text_simple_shape PASSED [100%]

============================== 7 passed in 1.68

0

In [11]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)

torch.Size([2, 16, 300])


## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [12]:
run_pytest("tests/test_train.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_train.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W14-gpt-lab
plugins: typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
collecting ... collected 5 items

tests/test_train.py::TestCalcLossBatch::test_calc_loss_batch_returns_scalar PASSED [ 20%]
tests/test_train.py::TestCalcLossLoader::test_calc_loss_loader_returns_float PASSED [ 40%]
tests/test_train.py::TestCheckpoint::test_save_load_checkpoint_restores_epoch_and_step PASSED [ 60%]
tests/test_train.py::TestGenerate::test_generate_shape PASSED            [ 80%]
tests/test_train.py::TestPlotLosses::test_plot_losses_callable PASSED    [100%]

============================== 5 passed in 9.58s ===============================


선택한 테스트를 통과했습니다.


0

In [ ]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    small_tokenizer = BPETokenizer(vocab_size=300)
    small_tokenizer.train(corpus[:5000])
    token_ids = small_tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)

smoke loss: 5.811628341674805


#### weight update까지 확인, parameter changed가 0보다 크면 실제 학습 update 되고 있는 것

In [ ]:
import torch
from dataset import create_dataloader
from model import GPTModel
from train import calc_loss_batch, calc_loss_loader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

context_length = 32
batch_size = 4

train_ids = small_tokenizer.encode(corpus[:50_000])
val_ids = small_tokenizer.encode(val_corpus[:10_000])

train_loader = create_dataloader(train_ids, context_length, batch_size=batch_size, shuffle=True)
val_loader = create_dataloader(val_ids, context_length, batch_size=batch_size, shuffle=False)

config = {
    "vocab_size": len(small_tokenizer.id_to_token),
    "context_length": context_length,
    "emb_dim": 32,
    "n_heads": 4,
    "n_layers": 1,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

model = GPTModel(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

inp, tgt = next(iter(train_loader))
before = next(model.parameters()).detach().clone()

optimizer.zero_grad()
loss = calc_loss_batch(inp, tgt, model, device)
loss.backward()
optimizer.step()

after = next(model.parameters()).detach()

print("one batch loss:", loss.item())
print("parameter changed:", (after - before).abs().sum().item())
print("train avg loss:", calc_loss_loader(train_loader, model, device, num_batches=2))
print("val avg loss:", calc_loss_loader(val_loader, model, device, num_batches=2))

device: cuda
one batch loss: 5.9354424476623535
parameter changed: 1.9541823863983154
train avg loss: 5.903980493545532
val avg loss: 5.868087291717529


### 진짜 사전 학습 실행

저장된 BPE vocabulary와 전체 NSMC LM corpus를 사용해 GPT를 사전 학습합니다. T4 기준 baseline 설정으로 시작하고, checkpoint와 실험 기록을 Google Drive에 복사합니다.


In [ ]:
# 진짜 사전 학습 실행
import json
import shutil
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from dataset import create_dataloader
from model import GPTModel
from train import train_model, calc_loss_loader

cell_start_time = time.perf_counter()

# cell 11에서 Drive vocabulary를 load한 tokenizer를 그대로 사용합니다.
# tokenizer가 없다면 먼저 BPE vocabulary load 셀을 실행하세요.
assert "tokenizer" in globals(), "먼저 BPE vocabulary load 셀을 실행해 tokenizer를 준비하세요."
assert len(tokenizer.id_to_token) == 3000, f"expected vocab size 3000, got {len(tokenizer.id_to_token)}"
assert "corpus" in globals() and len(corpus) > 0, "먼저 NSMC LM train corpus를 로드하세요."

# 전체 corpus로 최종 사전 학습을 시작합니다.
# 빠른 탐색이 필요하면 TRAIN_CHAR_LIMIT를 300_000 또는 500_000으로 낮춰서 먼저 비교하세요.
TRAIN_CHAR_LIMIT = None
VAL_CHAR_LIMIT = None

train_text = corpus if TRAIN_CHAR_LIMIT is None else corpus[:TRAIN_CHAR_LIMIT]
if "val_corpus" in globals() and val_corpus:
    val_text = val_corpus if VAL_CHAR_LIMIT is None else val_corpus[:VAL_CHAR_LIMIT]
else:
    split_idx = int(len(train_text) * 0.9)
    val_text = train_text[split_idx:]
    train_text = train_text[:split_idx]

context_length = 128
batch_size = 8
learning_rate = 3e-4
num_epochs = 3
eval_freq = 100
eval_iter = 20
ckpt_freq = 1
encode_each_line_with_bos_eos = True

config = {
    "vocab_size": len(tokenizer.id_to_token),
    "context_length": context_length,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

def encode_lm_text(text: str, add_bos_eos_per_line: bool) -> list[int]:
    if not add_bos_eos_per_line:
        return tokenizer.encode(text)

    token_ids = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            token_ids.extend(tokenizer.encode(line, add_bos_eos=True))
    return token_ids


cache_root = Path("/content/drive/MyDrive/W14-gpt-lab/cache")
if not cache_root.exists():
    cache_root = Path("cache")
cache_root.mkdir(parents=True, exist_ok=True)

bos_tag = "bos" if encode_each_line_with_bos_eos else "nobos"
train_limit_tag = "full" if TRAIN_CHAR_LIMIT is None else str(TRAIN_CHAR_LIMIT)
val_limit_tag = "full" if VAL_CHAR_LIMIT is None else str(VAL_CHAR_LIMIT)
cache_name = f"bpe{len(tokenizer.id_to_token)}_{bos_tag}_train{train_limit_tag}_val{val_limit_tag}.pt"
cache_path = cache_root / cache_name

encode_start_time = time.perf_counter()
if cache_path.exists():
    cached = torch.load(cache_path, map_location="cpu")
    train_ids = cached["train_ids"]
    val_ids = cached["val_ids"]
    print("loaded token id cache:", cache_path)
else:
    train_ids = encode_lm_text(train_text, encode_each_line_with_bos_eos)
    val_ids = encode_lm_text(val_text, encode_each_line_with_bos_eos)
    torch.save({"train_ids": train_ids, "val_ids": val_ids}, cache_path)
    print("saved token id cache:", cache_path)
encode_elapsed = time.perf_counter() - encode_start_time

train_loader = create_dataloader(
    train_ids,
    context_length=context_length,
    batch_size=batch_size,
    shuffle=True,
)
val_loader = create_dataloader(
    val_ids,
    context_length=context_length,
    batch_size=batch_size,
    shuffle=False,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("train chars:", len(train_text), "val chars:", len(val_text))
print("train tokens:", len(train_ids), "val tokens:", len(val_ids))
print("train batches:", len(train_loader), "val batches:", len(val_loader))
print("config:", config)
print("batch_size:", batch_size, "learning_rate:", learning_rate, "num_epochs:", num_epochs)
print("encode_each_line_with_bos_eos:", encode_each_line_with_bos_eos)
print("encode_time_sec:", round(encode_elapsed, 2))
print("encode_time_min:", round(encode_elapsed / 60, 2))

model = GPTModel(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

start_time = time.perf_counter()
train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=eval_freq,
    eval_iter=eval_iter,
    start_context="이 영화는",
    tokenizer=tokenizer,
    ckpt_freq=ckpt_freq,
)
elapsed = time.perf_counter() - start_time

print("training_loop_time_sec:", round(elapsed, 2))
print("training_loop_time_min:", round(elapsed / 60, 2))
final_full_val_loss = calc_loss_loader(val_loader, model, device, num_batches=None)
total_elapsed = time.perf_counter() - cell_start_time

print("best_train_loss:", min(train_losses) if train_losses else None)
print("best_val_loss_estimate:", min(val_losses) if val_losses else None)
print("final_train_loss_estimate:", train_losses[-1] if train_losses else None)
print("final_val_loss_estimate:", val_losses[-1] if val_losses else None)
print("final_full_val_loss:", final_full_val_loss)
print("total_cell_time_sec:", round(total_elapsed, 2))
print("total_cell_time_min:", round(total_elapsed / 60, 2))

eval_steps = [(i + 1) * eval_freq for i in range(len(train_losses))]
eval_epochs = [((step - 1) // len(train_loader)) + 1 for step in eval_steps]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(eval_steps, train_losses, marker="o", label="Train")
ax.plot(eval_steps, val_losses, marker="o", label="Val")
ax.set_xlabel("Global step / Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training / Validation Loss")
ax.legend()
ax.grid(alpha=0.25)

if eval_steps:
    tick_count = min(8, len(eval_steps))
    tick_indices = sorted(set(round(i * (len(eval_steps) - 1) / max(1, tick_count - 1)) for i in range(tick_count)))
    tick_values = [eval_steps[i] for i in tick_indices]
    tick_labels = [f"{eval_steps[i]}\nE{eval_epochs[i]}" for i in tick_indices]
    ax.set_xticks(tick_values)
    ax.set_xticklabels(tick_labels)

fig.tight_layout()
local_plot_path = Path("loss_curve.png")
fig.savefig(local_plot_path, dpi=150, bbox_inches="tight")
plt.show()

# Colab 런타임 저장소의 checkpoints/는 사라질 수 있으므로 Drive에도 복사합니다.
drive_root = Path("/content/drive/MyDrive/W14-gpt-lab")
if drive_root.exists():
    bos_tag = "bos" if encode_each_line_with_bos_eos else "nobos"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = (
        f"{timestamp}_pretrain"
        f"_ctx{context_length}"
        f"_{bos_tag}"
        f"_emb{config['emb_dim']}"
        f"_heads{config['n_heads']}"
        f"_layers{config['n_layers']}"
        f"_drop{config['drop_rate']}"
        f"_bs{batch_size}"
        f"_lr{learning_rate:g}"
        f"_ep{num_epochs}"
    )
    drive_run_dir = drive_root / "runs" / run_name

    suffix = 1
    while drive_run_dir.exists():
        drive_run_dir = drive_root / "runs" / f"{run_name}_repeat{suffix}"
        suffix += 1

    drive_ckpt_dir = drive_run_dir / "checkpoints"
    drive_ckpt_dir.mkdir(parents=True, exist_ok=False)

    drive_plot_path = drive_run_dir / "loss_curve.png"
    if local_plot_path.exists():
        shutil.copy2(local_plot_path, drive_plot_path)

    for epoch in range(1, num_epochs + 1):
        ckpt_path = Path("checkpoints") / f"ckpt_epoch_{epoch}.pt"
        if ckpt_path.exists():
            shutil.copy2(ckpt_path, drive_ckpt_dir / ckpt_path.name)

    summary = {
        "run_name": run_name,
        "run_dir": str(drive_run_dir),
        "train_chars": len(train_text),
        "val_chars": len(val_text),
        "train_tokens": len(train_ids),
        "val_tokens": len(val_ids),
        "token_cache_path": str(cache_path),
        "train_batches": len(train_loader),
        "val_batches": len(val_loader),
        "config": config,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "num_epochs": num_epochs,
        "eval_freq": eval_freq,
        "eval_iter": eval_iter,
        "ckpt_freq": ckpt_freq,
        "encode_each_line_with_bos_eos": encode_each_line_with_bos_eos,
        "encode_time_sec": round(encode_elapsed, 2),
        "training_loop_time_sec": round(elapsed, 2),
        "total_cell_time_sec": round(total_elapsed, 2),
        "train_losses": train_losses,
        "val_losses": val_losses,
        "eval_steps": eval_steps,
        "eval_epochs": eval_epochs,
        "loss_curve_path": str(drive_plot_path),
        "best_train_loss": min(train_losses) if train_losses else None,
        "best_val_loss_estimate": min(val_losses) if val_losses else None,
        "final_train_loss_estimate": train_losses[-1] if train_losses else None,
        "final_val_loss_estimate": val_losses[-1] if val_losses else None,
        "final_full_val_loss": final_full_val_loss,
    }

    with open(drive_run_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("saved run summary:", drive_run_dir / "summary.json")
    print("saved loss curve:", drive_plot_path)
    print("saved checkpoints:", drive_ckpt_dir)
else:
    print("Drive path not found. Checkpoints remain in local ./checkpoints")


### Optuna 하이퍼파라미터 짧은 탐색

전체 학습 전에 작은 corpus 조각으로 여러 조합을 빠르게 비교합니다. 최종 학습 결과가 아니라 후보를 좁히기 위한 탐색입니다.


In [ ]:
# Optuna 하이퍼파라미터 짧은 탐색
import json
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from dataset import create_dataloader
from model import GPTModel
from train import calc_loss_batch, calc_loss_loader

try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])
    import optuna

assert "tokenizer" in globals(), "먼저 BPE vocabulary load 셀을 실행해 tokenizer를 준비하세요."
assert len(tokenizer.id_to_token) == 3000, f"expected vocab size 3000, got {len(tokenizer.id_to_token)}"
assert "corpus" in globals() and len(corpus) > 0, "먼저 NSMC LM train corpus를 로드하세요."

# 탐색 비용을 줄이기 위한 데이터/반복 설정입니다. 최종 학습은 별도 셀에서 전체 corpus로 돌리세요.
OPTUNA_TRAIN_CHAR_LIMIT = 300_000
OPTUNA_VAL_CHAR_LIMIT = 30_000
OPTUNA_NUM_EPOCHS = 3
OPTUNA_N_TRIALS = 12
OPTUNA_TIMEOUT_SEC = None
OPTUNA_EVAL_ITER = 10
OPTUNA_SEED = 42

# 과제에서 제시한 후보군 + 레포 코드상 바로 튜닝 가능한 후보군입니다.
# 후보를 줄이면 탐색이 빨라지고, 늘리면 더 넓게 보지만 시간이 급격히 증가합니다.
OPTUNA_SEARCH_SPACE = {
    # 과제 후보
    "batch_size": [2, 4, 8, 16],
    "drop_rate": [0.0, 0.1, 0.2],
    "learning_rate": [1e-4, 3e-4, 5e-4],
    "context_length": [64, 128],
    "n_layers": [1, 2, 4],
    "emb_dim": [64, 128, 192],

    # 레포 코드에서 추가로 바로 실험 가능한 값
    "weight_decay": [0.0, 0.01, 0.1],
    "encode_each_line_with_bos_eos": [False, True],
    "n_heads": [4, 8],
    "qkv_bias": [False, True],

    # stride는 GPTDataset이 다음 chunk를 시작하는 간격입니다.
    # 1.0이면 stride=context_length, 0.5이면 절반씩 겹치게 잘라 학습 샘플 수가 늘어납니다.
    "stride_ratio": [1.0, 0.5],
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

search_train_text = corpus[:OPTUNA_TRAIN_CHAR_LIMIT]
if "val_corpus" in globals() and val_corpus:
    search_val_text = val_corpus[:OPTUNA_VAL_CHAR_LIMIT]
else:
    split_idx = int(len(search_train_text) * 0.9)
    search_val_text = search_train_text[split_idx:]
    search_train_text = search_train_text[:split_idx]


def encode_lm_text_for_search(text: str, add_bos_eos_per_line: bool) -> list[int]:
    if not add_bos_eos_per_line:
        return tokenizer.encode(text)

    token_ids = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            token_ids.extend(tokenizer.encode(line, add_bos_eos=True))
    return token_ids


cache_root = Path("/content/drive/MyDrive/W14-gpt-lab/cache")
if not cache_root.exists():
    cache_root = Path("cache")
cache_root.mkdir(parents=True, exist_ok=True)

_token_id_cache: dict[bool, tuple[list[int], list[int], Path, float]] = {}


def get_search_token_ids(add_bos_eos_per_line: bool) -> tuple[list[int], list[int], Path, float]:
    if add_bos_eos_per_line in _token_id_cache:
        return _token_id_cache[add_bos_eos_per_line]

    bos_tag = "bos" if add_bos_eos_per_line else "nobos"
    cache_name = (
        f"optuna_bpe{len(tokenizer.id_to_token)}"
        f"_{bos_tag}"
        f"_train{OPTUNA_TRAIN_CHAR_LIMIT}"
        f"_val{OPTUNA_VAL_CHAR_LIMIT}.pt"
    )
    cache_path = cache_root / cache_name

    encode_start = time.perf_counter()
    if cache_path.exists():
        cached = torch.load(cache_path, map_location="cpu")
        train_ids = cached["train_ids"]
        val_ids = cached["val_ids"]
        print("loaded token id cache:", cache_path)
    else:
        train_ids = encode_lm_text_for_search(search_train_text, add_bos_eos_per_line)
        val_ids = encode_lm_text_for_search(search_val_text, add_bos_eos_per_line)
        torch.save({"train_ids": train_ids, "val_ids": val_ids}, cache_path)
        print("saved token id cache:", cache_path)

    encode_elapsed = time.perf_counter() - encode_start
    _token_id_cache[add_bos_eos_per_line] = (train_ids, val_ids, cache_path, encode_elapsed)
    return _token_id_cache[add_bos_eos_per_line]


print("search train chars:", len(search_train_text), "val chars:", len(search_val_text))
print("search space:", OPTUNA_SEARCH_SPACE)


def suggest_categorical(trial: optuna.Trial, name: str):
    return trial.suggest_categorical(name, OPTUNA_SEARCH_SPACE[name])


def objective(trial: optuna.Trial) -> float:
    torch.manual_seed(OPTUNA_SEED)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    context_length = suggest_categorical(trial, "context_length")
    batch_size = suggest_categorical(trial, "batch_size")
    learning_rate = suggest_categorical(trial, "learning_rate")
    emb_dim = suggest_categorical(trial, "emb_dim")
    n_layers = suggest_categorical(trial, "n_layers")
    drop_rate = suggest_categorical(trial, "drop_rate")
    weight_decay = suggest_categorical(trial, "weight_decay")
    encode_each_line_with_bos_eos = suggest_categorical(trial, "encode_each_line_with_bos_eos")
    n_heads = suggest_categorical(trial, "n_heads")
    qkv_bias = suggest_categorical(trial, "qkv_bias")
    stride_ratio = suggest_categorical(trial, "stride_ratio")

    if emb_dim % n_heads != 0:
        raise optuna.TrialPruned(f"emb_dim {emb_dim} is not divisible by n_heads {n_heads}")

    stride = max(1, int(context_length * stride_ratio))
    train_ids, val_ids, cache_path, encode_elapsed = get_search_token_ids(encode_each_line_with_bos_eos)

    config = {
        "vocab_size": len(tokenizer.id_to_token),
        "context_length": context_length,
        "emb_dim": emb_dim,
        "n_heads": n_heads,
        "n_layers": n_layers,
        "drop_rate": drop_rate,
        "qkv_bias": qkv_bias,
    }

    train_loader = create_dataloader(
        train_ids,
        context_length=context_length,
        batch_size=batch_size,
        stride=stride,
        shuffle=True,
    )
    val_loader = create_dataloader(
        val_ids,
        context_length=context_length,
        batch_size=batch_size,
        stride=stride,
        shuffle=False,
    )

    trial.set_user_attr("train_tokens", len(train_ids))
    trial.set_user_attr("val_tokens", len(val_ids))
    trial.set_user_attr("train_batches", len(train_loader))
    trial.set_user_attr("val_batches", len(val_loader))
    trial.set_user_attr("stride", stride)
    trial.set_user_attr("token_cache_path", str(cache_path))
    trial.set_user_attr("encode_time_sec", round(encode_elapsed, 2))

    model = GPTModel(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    best_val_loss = float("inf")
    try:
        for epoch in range(OPTUNA_NUM_EPOCHS):
            model.train()
            for input_batch, target_batch in train_loader:
                optimizer.zero_grad()
                loss = calc_loss_batch(input_batch, target_batch, model, device)
                loss.backward()
                optimizer.step()

            val_loss = calc_loss_loader(val_loader, model, device, num_batches=OPTUNA_EVAL_ITER)
            best_val_loss = min(best_val_loss, val_loss)
            trial.report(val_loss, step=epoch)

            print(
                f"trial {trial.number} epoch {epoch + 1}/{OPTUNA_NUM_EPOCHS}: "
                f"val_loss={val_loss:.4f}, best={best_val_loss:.4f}, params={trial.params}"
            )

            if trial.should_prune():
                raise optuna.TrialPruned()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            raise optuna.TrialPruned("CUDA out of memory")
        raise
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return best_val_loss


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1),
)

search_start = time.perf_counter()
study.optimize(objective, n_trials=OPTUNA_N_TRIALS, timeout=OPTUNA_TIMEOUT_SEC)
search_elapsed = time.perf_counter() - search_start

print("best value:", study.best_value)
print("best params:", study.best_params)
print("optuna_search_time_min:", round(search_elapsed / 60, 2))

completed_trials = [t for t in study.trials if t.value is not None]
trial_numbers = [t.number for t in completed_trials]
trial_values = [t.value for t in completed_trials]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(trial_numbers, trial_values, marker="o")
ax.set_xlabel("Trial")
ax.set_ylabel("Best validation loss")
ax.set_title("Optuna Search Results")
ax.grid(alpha=0.25)
fig.tight_layout()

local_optuna_plot_path = Path("optuna_trials.png")
fig.savefig(local_optuna_plot_path, dpi=150, bbox_inches="tight")
plt.show()

runs_root = Path("/content/drive/MyDrive/W14-gpt-lab/runs")
if not runs_root.exists():
    runs_root = Path("runs")
runs_root.mkdir(parents=True, exist_ok=True)

optuna_run_name = datetime.now().strftime("%Y%m%d_%H%M%S_optuna_search")
optuna_run_dir = runs_root / optuna_run_name
optuna_run_dir.mkdir(parents=True, exist_ok=False)

with open(optuna_run_dir / "optuna_summary.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "best_value": study.best_value,
            "best_params": study.best_params,
            "search_space": OPTUNA_SEARCH_SPACE,
            "n_trials": OPTUNA_N_TRIALS,
            "num_epochs_per_trial": OPTUNA_NUM_EPOCHS,
            "train_char_limit": OPTUNA_TRAIN_CHAR_LIMIT,
            "val_char_limit": OPTUNA_VAL_CHAR_LIMIT,
            "eval_iter": OPTUNA_EVAL_ITER,
            "search_time_sec": round(search_elapsed, 2),
            "trials": [
                {
                    "number": t.number,
                    "value": t.value,
                    "state": str(t.state),
                    "params": t.params,
                    "user_attrs": t.user_attrs,
                }
                for t in study.trials
            ],
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

if local_optuna_plot_path.exists():
    shutil.copy2(local_optuna_plot_path, optuna_run_dir / "optuna_trials.png")

try:
    study.trials_dataframe().to_csv(optuna_run_dir / "optuna_trials.csv", index=False)
except Exception as e:
    print("trials_dataframe save skipped:", e)

print("saved optuna summary:", optuna_run_dir / "optuna_summary.json")
print("saved optuna plot:", optuna_run_dir / "optuna_trials.png")
print("Use best_params in the real pretraining cell, then rerun on a larger corpus.")


## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [ ]:
run_pytest("tests/")